# 11c — Jeux globaux : sélection d'équilibre sous information incomplète (jumeau C#)

**Navigation** : [<< 11-BayesianGames (twin C#)](GameTheory-11-BayesianGames-CSharp.ipynb) | [Twin Python](GameTheory-11c-Global-Games-Python.ipynb) | [Index](README.md) | [12-ReputationGames (twin C#) >>](GameTheory-12-ReputationGames-CSharp.ipynb)

**Jumeau C#** de [GameTheory-11c-Global-Games-Python](GameTheory-11c-Global-Games-Python.ipynb) — même modèle, même pédagogie, implémenté en C# **sans dépendance NuGet** : la fonction de répartition et la densité de la normale standard sont calculées directement (CDF par la fonction d'erreur `erf`, Abramowitz & Stegun 7.1.26), et les figures sont rendues en **SVG natif** par le socle zero-dépendance [`SvgChartHelper`](../Probas/Infer/SvgChartHelper.cs).

Un jeu de coordination a **plusieurs équilibres** : deux issues stables, et rien, dans le jeu lui-même, pour prédire laquelle va se réaliser. Les **jeux globaux** (édifice bayésien de Harsanyi, puis Carlsson & van Damme (1993), puis Morris & Shin (1998)) lèvent cette indétermination **sans ajouter d'arbre ni d'action observable** : on donne aux joueurs une *information incomplète partagée* — un état commun qu'ils ne connaissent que par un **signal bruité** (un *tremble* privé). L'équilibre de Nash bayésien devient **unique** (tremble sous le niveau critique), et quand le tremble s'amenuise, il **sélectionne** l'issue cohérente avec l'état fondamental.

In [1]:
#load "../Probas/Infer/SvgChartHelper.cs"

using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
Console.WriteLine("Environnement pret : normale standard calculee directement (zero-dependance)  |  SvgChartHelper charge");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Environnement pret : normale standard calculee directement (zero-dependance)  |  SvgChartHelper charge


In [2]:
// --- Le modele des jeux globaux (socle du carnet, zero-dependance) ---
// m = 1.0 : benefice de coordination : -m si je suis seul, +m si l'autre entre.
// theta ~ N(0,1) ; signal x_i = theta + eps_i, eps_i ~ N(0, sigma^2).
public static class GT11
{
    public const double M = 1.0;

    // --- normale standard, calculee directement (CDF par la fonction d'erreur) ---
    //   Phi(x) = 0.5*(1 + erf(x/sqrt(2))) ; phi(x) = exp(-x^2/2)/sqrt(2*pi).
    // erf : approximation d'Abramowitz & Stegun 7.1.26, |erreur| < 1.5e-7.
    static double Erf(double x)
    {
        bool neg = x < 0; x = Math.Abs(x);
        double t = 1.0 / (1.0 + 0.3275911 * x);
        double q = 1.061405429;
        q = q * t - 1.453152027;
        q = q * t + 1.421413741;
        q = q * t - 0.284496736;
        q = q * t + 0.254829592;
        q = q * t;
        double y = 1.0 - q * Math.Exp(-x * x);
        return neg ? -y : y;
    }
    public static double Phi(double x) => 0.5 * (1.0 + Erf(x / Math.Sqrt(2.0)));
    public static double PhiPdf(double x) => Math.Exp(-0.5 * x * x) / Math.Sqrt(2.0 * Math.PI);

    // --- croyances (moyenne normale) ---
    // E[theta | x] — croyance conditionnelle.
    public static double Mu(double x, double sigma) => x / (1.0 + sigma * sigma);
    public static double VarGivenX(double sigma) => sigma * sigma / (1.0 + sigma * sigma);
    // P(l'autre entre | x) = P(x_j >= c | x), sous la strategie seuil de l'autre.
    public static double POtherEnters(double x, double c, double sigma)
    {
        double sd = Math.Sqrt(sigma * sigma + VarGivenX(sigma));
        return Phi((Mu(x, sigma) - c) / sd);
    }
    // E[gain d'entrer | x], sachant que l'autre entre si x_j >= c.
    public static double PayoffEnter(double x, double c, double sigma)
        => Mu(x, sigma) - M + 2.0 * M * POtherEnters(x, c, sigma);

    // Le joueur au signal x = c est indifferent entre entrer et rester :
    //   E[theta | c] - m + 2m*P(l'autre entre | c) = 0.
    public static double Indifference(double c, double sigma) => PayoffEnter(c, c, sigma);

    // Racine de l'indifference ; bisection. Renvoie le seuil BNE unique c*(sigma) ;
    // au-dela du tremble critique, rend deterministiquement la PLUS GRANDE racine ;
    // null si aucun changement de signe sur [lo, hi].
    public static double? SolveCutoff(double sigma, double lo = -8.0, double hi = 8.0)
    {
        double Hlo = Indifference(lo, sigma), Hhi = Indifference(hi, sigma);
        if (Hlo * Hhi > 0) return null;
        for (int i = 0; i < 200; i++)
        {
            double mid = 0.5 * (lo + hi);
            if (Indifference(mid, sigma) > 0) hi = mid; else lo = mid;
        }
        return 0.5 * (lo + hi);
    }

    // Equivalent de scipy.optimize.brentq : bisection bornee (f continue, changement de signe).
    public static double Bisect(Func<double, double> f, double lo, double hi, int iters = 200)
    {
        double flo = f(lo), fhi = f(hi);
        if (flo * fhi > 0)
            throw new InvalidOperationException($"Bisect : pas de changement de signe sur [{lo}, {hi}]");
        for (int i = 0; i < iters; i++)
        {
            double mid = 0.5 * (lo + hi);
            if (f(mid) > 0) hi = mid; else lo = mid;
        }
        return 0.5 * (lo + hi);
    }

    // H(c) en forme close : H(c) = c/s - m + 2m*Phi(-k*c), s = 1+sigma^2, k = sigma/sqrt(s*(s+1)).
    public static (double s, double k) Coeffs(double sigma)
    {
        double s = 1.0 + sigma * sigma;
        double k = sigma / Math.Sqrt(s * (s + 1.0));
        return (s, k);
    }
    public static double HFerme(double c, double sigma)
    {
        var (s, k) = Coeffs(sigma);
        return c / s - M + 2.0 * M * Phi(-k * c);
    }
    // H'(0) = 1/s - 2*m*k*phi(0) ; phi est maximale en 0, donc H'(c) >= H'(0).
    public static double HPrime0(double sigma)
    {
        var (s, k) = Coeffs(sigma);
        return 1.0 / s - 2.0 * M * k * PhiPdf(0.0);
    }
    // Tremble critique : racine de 4 m^2 u^2 + (4 m^2 - 2 pi) u - 4 pi = 0, u = sigma^2.
    public static double TrembleCritique(double m = M)
    {
        double u = (2 * Math.PI - 4 * m * m
                    + Math.Sqrt(16 * m * m * m * m + 48 * Math.PI * m * m + 4 * Math.PI * Math.PI))
                   / (8 * m * m);
        return Math.Sqrt(u);
    }

    // Racines distinctes de H, cherchees sur [0, hi] ; l'imparite donne les negatives.
    // On demarre le scan a i=1 : c=0 est TOUJOURS une racine (ajoutee explicitement),
    // et Indifference(0) n'est pas bit-a-bit nulle (chemin croyances vs forme close),
    // un scan demarrant a i=0 y detecterait un faux changement de signe juste au-dessus de 0.
    public static List<double> SeuilsH(double sigma, double hi = 40.0, int n = 16001)
    {
        var grille = new double[n];
        var vals = new double[n];
        for (int i = 0; i < n; i++) { grille[i] = hi * i / (n - 1); vals[i] = Indifference(grille[i], sigma); }
        var positifs = new List<double>();
        for (int i = 1; i < n - 1; i++)
            if (vals[i] * vals[i + 1] < 0.0)
                positifs.Add(Bisect(c => Indifference(c, sigma), grille[i], grille[i + 1]));
        var res = new List<double>();
        if (positifs.Count > 0)
        {
            foreach (var p in positifs) res.Add(-p);
            res.Add(0.0);
            foreach (var p in positifs) res.Add(p);
        }
        else res.Add(0.0);
        return res;
    }
}

Console.WriteLine("Socle GT11 defini : Phi/PhiPdf (normale directe) + Mu/POtherEnters/PayoffEnter/Indifference/" +
    "SolveCutoff/Bisect/Coeffs/HFerme/HPrime0/TrembleCritique/SeuilsH  |  Phi(1) = " + GT11.Phi(1.0).ToString("F6"));

Socle GT11 defini : Phi/PhiPdf (normale directe) + Mu/POtherEnters/PayoffEnter/Indifference/SolveCutoff/Bisect/Coeffs/HFerme/HPrime0/TrembleCritique/SeuilsH  |  Phi(1) = 0,841345


## 1. L'équilibre de Nash bayésien est une stratégie de seuil

Dans ce jeu de coordination à **complémentarité**, le gain d'entrer est **croissant** en son propre signal `x` : l'entrée est d'autant plus attractive que le signal est haut. Un équilibre est donc une **stratégie seuil** — entrer si `x_i ≥ c` — où le joueur au signal `x = c` est **indifférent** :

```
E[theta | c]  -  m  +  2m · P(l'autre entre | c)  =  0
```

La bisection de cette indifférence donne le seuil BNE `c*(sigma)`.

In [3]:
// Le seuil BNE c*(sigma) pour une gamme de trembles : plus le signal est fiable
// (sigma petit), plus le seuil s'approche de 0 (le fondamental).
Console.WriteLine($"seuil BNE c*(sigma),  m = {GT11.M}");
foreach (double s in new double[] { 1.0, 0.5, 0.3, 0.1, 0.05, 0.01, 0.001 })
{
    double? r = GT11.SolveCutoff(s);
    Console.WriteLine($"  sigma={s,-6:0.###}  c*={(r ?? double.NaN):F4}");
}

seuil BNE c*(sigma),  m = 1


  sigma=1       c*=-0,0000


  sigma=0,5     c*=-0,0000


  sigma=0,3     c*=-0,0000


  sigma=0,1     c*=-0,0000


  sigma=0,05    c*=-0,0000


  sigma=0,01    c*=-0,0000


  sigma=0,001   c*=-0,0000


In [4]:
// --- verification de la MEILLEURE REPONSE de part et d'autre du seuil ---
// Trouver une racine de l'indifference ne suffit pas : il faut que la strategie seuil
// soit une MEILLEURE REPONSE a l'autre qui joue le meme seuil.
double sigma = 0.1;
double c = GT11.SolveCutoff(sigma).Value;
double below = GT11.PayoffEnter(c - 0.01, c, sigma);   // juste sous le seuil : prescrit Rester
double above = GT11.PayoffEnter(c + 0.01, c, sigma);    // juste au-dessus  : prescrit Entrer
double deriv = (above - below) / 0.02;                  // derive du gain en son signal, au seuil
const string Sgn6 = "+0.000000;-0.000000;0.000000";
const string Sgn4 = "+0.0000;-0.0000;0.0000";
Console.WriteLine($"sigma={sigma},  seuil c*={c:F6}");
Console.WriteLine("  juste SOUS le seuil  : gain d'entrer = " + below.ToString(Sgn6) + "  (doit etre < 0 -> rester optimal)");
Console.WriteLine("  juste AU-DESSUS      : gain d'entrer = " + above.ToString(Sgn6) + "  (doit etre > 0 -> entrer optimal)");
Console.WriteLine("  derive du gain en x au seuil = " + deriv.ToString(Sgn4) + "  (> 0 : meilleure reponse croissante)");
if (!(below < 0.0 && above > 0.0))
    throw new InvalidOperationException("MEILLEURE REPONSE violee : ce n'est pas un BNE de seuil");
Console.WriteLine("-> les deux inegalites de meilleure reponse tiennent : le seuil est bien un BNE.");

sigma=0,1,  seuil c*=-0,000000


  juste SOUS le seuil  : gain d'entrer = -0,065854  (doit etre < 0 -> rester optimal)


  juste AU-DESSUS      : gain d'entrer = +0,065854  (doit etre > 0 -> entrer optimal)


  derive du gain en x au seuil = +6,5854  (> 0 : meilleure reponse croissante)


-> les deux inegalites de meilleure reponse tiennent : le seuil est bien un BNE.


## 2. La sélection : le seuil suit le fondamental, l'issue suit l'état

La figure 1 trace le seuil d'équilibre `c*(sigma)` en fonction du tremble : il **coïncide avec le fondamental** (la moyenne `0` de `theta`) pour tout niveau de bruit sous le critique. La figure 2 montre l'effet sélectionneur : la probabilité que **les deux** joueurs entrent, conditionnelle à l'état `theta`, saute en `1{theta > 0}` quand le tremble s'amenuise — c'est l'issue de coordination cohérente avec l'état fondamental qui l'emporte.

In [5]:
// Figure 1 — le seuil d'équilibre coïncide avec le fondamental (moyenne 0 de theta)
var sigmas = new List<double>();
for (int i = 0; i < 60; i++) sigmas.Add(0.001 + (1.0 - 0.001) * i / 59.0);
var cstar = sigmas.Select(s => GT11.SolveCutoff(s).Value).ToArray();
var zeref = sigmas.Select(_ => 0.0).ToArray();
display(SvgChartHelper.Overlay(
    "Le seuil d'équilibre coïncide avec le fondamental",
    "tremble sigma (bruit du signal)",
    "seuil BNE c*(sigma)",
    new List<SvgSeries> {
        new SvgSeries("seuil BNE c*(sigma)", sigmas.ToArray(), cstar, TraceStyle.LineMarkers, "#4C72B0"),
        new SvgSeries("fondamental = 0 (moyenne de theta)", sigmas.ToArray(), zeref, TraceStyle.Line, "#888888")
    }));

Le seuil d'équilibre coïncide avec le fondamental -0 -0 -0 -0 0 0.001 0.251 0.501 0.75 1 tremble sigma (bruit du signal) seuil BNE c*(sigma) <polyline points='52,118.538 64.746,119.165 77.492,119.842 90.237,120.569 102.983,121.35 115.729,122.185 128.475,123.077 141.22,124.028 153.966,125.04 166.712,126.115 179.458,127.256 192.203,128.466 204.949,129.746 217.695,131.101 230.441,132.532 243.186,134.044 255.932,135.639 268.678,137.321 281.424,139.094 294.169,140.962 306.915,142.93 319.661,145.001 332.407,147.18 345.153,149.473 357.898,151.885 370.644,154.423 383.39,157.091 396.136,159.898 408.881,162.849 421.627,165.953 434.373,169.218 447.119,172.653 459.864,176.267 472.61,180.071 485.356,184.075 498.102,188.291 510.847,192.733 523.593,197.414 536.339,202.35 549.085,207.556 561.831,213.051 574.576,218.855 587.322,224.989 600.068,231.475 612.814,238.341 625.559,245.614 638.305,253.326 651.051,261.51 663.797,270.206 676.542,279.455 689.288,289.304 702.034,299.807 714.78,311.023 727.525,323.018 740.271,335.867 753.017,349.655 765.763,364.48 778.508,380.45 791.254,397.694 804,416.357' fill='none' stroke='#4C72B0' stroke-width='2'/> <polyline points='52,55.643 64.746,55.643 77.492,55.643 90.237,55.643 102.983,55.643 115.729,55.643 128.475,55.643 141.22,55.643 153.966,55.643 166.712,55.643 179.458,55.643 192.203,55.643 204.949,55.643 217.695,55.643 230.441,55.643 243.186,55.643 255.932,55.643 268.678,55.643 281.424,55.643 294.169,55.643 306.915,55.643 319.661,55.643 332.407,55.643 345.153,55.643 357.898,55.643 370.644,55.643 383.39,55.643 396.136,55.643 408.881,55.643 421.627,55.643 434.373,55.643 447.119,55.643 459.864,55.643 472.61,55.643 485.356,55.643 498.102,55.643 510.847,55.643 523.593,55.643 536.339,55.643 549.085,55.643 561.831,55.643 574.576,55.643 587.322,55.643 600.068,55.643 612.814,55.643 625.559,55.643 638.305,55.643 651.051,55.643 663.797,55.643 676.542,55.643 689.288,55.643 702.034,55.643 714.78,55.643 727.525,55.643 740.271,55.643 753.017,55.643 765.763,55.643 778.508,55.643 791.254,55.643 804,55.643' fill='none' stroke='#888888' stroke-width='2'/> seuil BNE c*(sigma) fondamental = 0 (moyenne de theta)

In [6]:
// Figure 2 — la probabilité de coordonnaison suit l'état theta
double sig2 = 0.3;
double cst2 = GT11.SolveCutoff(sig2).Value;
var thetas = new List<double>();
for (int i = 0; i < 80; i++) thetas.Add(-3.0 + 6.0 * i / 79.0);
var pboth = thetas.Select(t => {
    double p = GT11.Phi((t - cst2) / sig2);
    return p * p;
}).ToArray();
string cst2s = cst2.ToString("0.000");
display(SvgChartHelper.Overlay(
    "La coordonnaison suit l'état : P(les deux entrent | theta)",
    "état fundamental theta",
    "P(les deux entrent | theta)",
    new List<SvgSeries> {
        new SvgSeries("P(les deux entrent | theta)", thetas.ToArray(), pboth, TraceStyle.Line, "#DD8452")
    }));
Console.WriteLine($"sigma={sig2}  c*={cst2s}  ->  P(les deux entrent | theta=0) = {(GT11.Phi((0.0 - cst2) / sig2) * GT11.Phi((0.0 - cst2) / sig2)).ToString("0.0000")}");

La coordonnaison suit l'état : P(les deux entrent | theta) -0.06 0.22 0.5 0.78 1.06 -3 -1.5 0 1.5 3 état fundamental theta P(les deux entrent | theta) <polyline points='52,416.357 61.519,416.357 71.038,416.357 80.557,416.357 90.076,416.357 99.595,416.357 109.114,416.357 118.633,416.357 128.152,416.357 137.671,416.357 147.19,416.357 156.709,416.357 166.228,416.357 175.747,416.357 185.266,416.357 194.785,416.357 204.304,416.357 213.823,416.357 223.342,416.357 232.861,416.357 242.38,416.357 251.899,416.357 261.418,416.357 270.937,416.357 280.456,416.357 289.975,416.357 299.494,416.357 309.013,416.357 318.532,416.356 328.051,416.352 337.57,416.334 347.089,416.268 356.608,416.058 366.127,415.458 375.646,413.938 385.165,410.512 394.684,403.637 404.203,391.332 413.722,371.646 423.241,343.431 432.759,307.096 442.278,264.923 451.797,220.639 461.316,178.398 470.835,141.636 480.354,112.308 489.873,90.762 499.392,76.121 508.911,66.882 518.43,61.452 527.949,58.471 537.468,56.94 546.987,56.203 556.506,55.871 566.025,55.73 575.544,55.674 585.063,55.654 594.582,55.646 604.101,55.644 613.62,55.643 623.139,55.643 632.658,55.643 642.177,55.643 651.696,55.643 661.215,55.643 670.734,55.643 680.253,55.643 689.772,55.643 699.291,55.643 708.81,55.643 718.329,55.643 727.848,55.643 737.367,55.643 746.886,55.643 756.405,55.643 765.924,55.643 775.443,55.643 784.962,55.643 794.481,55.643 804,55.643' fill='none' stroke='#DD8452' stroke-width='2'/> P(les deux entrent | theta)

sigma=0,3  c*=-0,000  ->  P(les deux entrent | theta=0) = 0,2500


## 3. L'unicité : la courbe d'indifférence traverse zéro une seule fois

Le seuil d'équilibre est l'origine (racine) de la **fonction d'indifférence** `indifference(c, sigma)`. Si cette fonction croît strictement à travers zéro — dérivée `> 0` au point d'annulation —, sa racine est **unique** : un seul seuil d'équilibre. On mesure la pente en **différences centrales** au zéro, pour trois niveaux de bruit du signal.

In [7]:
// unicité : pente de la fonction d'indifférence à travers son zéro
double[] sigmas_u = {0.1, 1.0, 2.0};
Console.WriteLine("sigma     pente d[indiff] au zero     verdict");
foreach (double s in sigmas_u)
{
    double d = (GT11.Indifference(0.01, s) - GT11.Indifference(-0.01, s)) / 0.02;
    string verdict = d > 0 ? "unique (1 seuil)" : "multiplicite (3 seuils)";
    Console.WriteLine($"{s:0.0}     {d:0.0000}                   {verdict}");
}

sigma     pente d[indiff] au zero     verdict


0,1     0,9341                   unique (1 seuil)


1,0     0,1743                   unique (1 seuil)


2,0     -0,0913                   multiplicite (3 seuils)


## 4. Le tremble critique : bifurcation entre unicité et multiplicité

La **dérivée de H à l'origine** `H'(0) = 1/s − 2·m·k·φ(0)` est le **discriminant d'unicité**. Tant qu'elle est **positive**, H est strictement croissante au zéro : une seule racine, un seul seuil. Elle **s'annule** au **tremble critique** `sigma_c` (forme close) — c'est là que naît une **racine triple** à l'origine, de terme cubique `A = m·φ(0)·k³/3`. Au-delà, `H'(0) < 0` : H s'enfonce en négatif près du zéro avant de remonter, et **trois seuils** coexistent (multiplicité d'équilibre). Le recensement `SeuilsH` le confirme : 1 racine sous le seuil, 3 au-delà.

In [8]:
// Le tremble critique : la derive a l'origine s'annule -> racine triple -> bifurcation
double sigc = GT11.TrembleCritique();
var (sc, kc) = GT11.Coeffs(sigc);
double A_c = GT11.M * GT11.PhiPdf(0.0) * kc * kc * kc / 3.0;
Console.WriteLine($"tremble critique sigma_c = {sigc:0.0000}   (forme close)");
Console.WriteLine($"H'(0) au tremble critique = {GT11.HPrime0(sigc):+0.000000;-0.000000;0.000000}   (doit etre ~0)");
Console.WriteLine($"coefficient racine triple  A = m*phi(0)*k^3/3 = {A_c:0.000000}");
Console.WriteLine();
Console.WriteLine("sigma     H'(0)          #seuils H    regime");
double[] sigs = {0.3, 0.5, 0.8, sigc, 1.0, 1.5, 2.0};
foreach (double s in sigs)
{
    double hp = GT11.HPrime0(s);
    int nroots = GT11.SeuilsH(s).Count;
    string regime = hp > 0 ? "unique" : (hp < 0 ? "multiple (3)" : "critique");
    Console.WriteLine($"{s:0.000}     {hp:+0.0000;-0.0000;0.0000}       {nroots}           {regime}");
}

tremble critique sigma_c = 1,4425   (forme close)


H'(0) au tremble critique = 0,000000   (doit etre ~0)


coefficient racine triple  A = m*phi(0)*k^3/3 = 0,008954


sigma     H'(0)          #seuils H    regime


0,300     +0,7588       1           unique


0,500     +0,5621       1           unique


0,800     +0,3030       1           unique


1,442     0,0000       3           critique


1,000     +0,1743       1           unique


1,500     -0,0143       3           multiple (3)


2,000     -0,0913       3           multiple (3)


## Exercices

Trois exercices pour consolider le modèle. Chacun est un **stub** à compléter (il renvoie `0.0` et affiche un message) — le carnet s'exécute de bout en bout même non complété. Les fonctions `GT11.*` du socle (cellule 2) sont à votre disposition.

In [9]:
// Exercice 1 — Le seuil d'équilibre
// Enonce : calcule le seuil BNE c*(sigma) en resolvant l'indifference du joueur marginal.
// Indice : reutilise GT11.SolveCutoff(sigma), ou bisecte GT11.Indifference(c, sigma) sur [0, 8].
public static class Exo1
{
    public static double SeuilBayesien(double sigma)
    {
        // TODO etudiant : retourner le seuil BNE c*(sigma)
        return 0.0;
    }
}
Console.WriteLine("Exercice 1 a completer :  c*(0.3) = " + Exo1.SeuilBayesien(0.3).ToString("F4"));

Exercice 1 a completer :  c*(0.3) = 0,0000


In [10]:
// Exercice 2 — La probabilité de coordonnaison
// Enonce : calcule P(les deux entrent | theta) pour un etat fundamental theta donne.
// Indice : chaque joueur entre si son signal x > c* ; comme x = theta + eps,
//           P(x > c* | theta) = Phi((theta - c*)/sigma) ; eleve au carre pour les deux joueurs.
public static class Exo2
{
    public static double ProbCoordonner(double theta, double sigma)
    {
        // TODO etudiant : retourner P(les deux entrent | theta)
        return 0.0;
    }
}
Console.WriteLine("Exercice 2 a completer :  P(les deux entrent | theta=1) = " + Exo2.ProbCoordonner(1.0, 0.3).ToString("F4"));

Exercice 2 a completer :  P(les deux entrent | theta=1) = 0,0000


In [11]:
// Exercice 3 — Le fondamental
// Enonce : renvoie le FONDAMENTAL — la moyenne de theta — que le seuil d'equilibre epouse.
// Indice : theta ~ N(0, 1) ; quel est E[theta] ? Le seuil c*(sigma) coincide-t-il avec lui ?
public static class Exo3
{
    public static double SeuilFondamental()
    {
        // TODO etudiant : retourner le fondamental (moyenne de theta)
        return 0.0;
    }
}
Console.WriteLine("Exercice 3 a completer :  fondamental = " + Exo3.SeuilFondamental().ToString("F4"));

Exercice 3 a completer :  fondamental = 0,0000


## À retenir

- **L'équilibre de Nash bayésien est une stratégie de seuil** : chaque joueur entre si son signal `x` dépasse un seuil `c*` commun, qui est la racine de sa fonction d'indifférence.
- **Le seuil suit le fondamental** : `c*(sigma)` coïncide avec la moyenne du fondamental (`0` ici), quel que soit le bruit `sigma` — l'équilibre « sélectionne » le bon état en moyenne.
- **L'issue suit l'état** : la probabilité que les deux entrent, `P(les deux entrent | theta)`, est une sigmoïde centrée sur le seuil : la coordination est probable quand le fondamental est favorable.
- **Unicité vs multiplicité, pilotées par le bruit** : le discriminant est `H'(0)`. Sous le **tremble critique** `sigma_c`, l'équilibre est **unique** ; au-delà, **trois seuils** coexistent (multiplicité) — d'où le besoin de **sélection d'équilibre** (parafin, tremble, perfection).
- **Jumeau C#** : tout le modèle tourne en **C# sans dépendance** — la normale standard est calculée directement par la fonction d'erreur `erf` (Abramowitz & Stegun 7.1.26), et les figures en **SVG natif** via `SvgChartHelper`.